# Phase 5-C1 — new registered Development attempt
Safe unsigned preparation by default. Holdout NO_ACCESS, market NO_GO; formal EXECUTE requires separate approval.


In [ ]:
MODE = "PREPARE"  # With no approved context: unsigned authorization packet only.
EXECUTION_CODE_SHA = "a4cc87e91dde6fbbba3b71e294224745a8d240f9"
CORE_SHA = "50ea3bfa42df7708dd3a1b21f17798c206be054e"
NEW_AUTHORIZATION_ID = "RA-IBKR_TSM-SMA520_P5C1-0002"
GOVERNANCE_SHA = ""  # New approved governance commit; the old snapshot is forbidden.
AUTHORIZATION_RECORD_SHA256 = ""  # Exact new approved record, not the old record.
RECOVERY_ARCHIVE = ""  # Existing reviewed Development recovery archive; no new conversion.
ORIGINAL_PREPARATION_ARCHIVE = ""  # Existing personal data-pin review evidence.
PREPARATION = ""  # New preparation printed after approved-authority PREPARE.
BIND_URL = ""  # New personal BIND_AUTHORIZATION comment.
TASK_URL = ""  # New personal APPROVE_TASK comment, distinct from BIND_URL.
CONFIRM_AUTHORIZATION = ""
CONFIRM_REMOTE_CLAIM = False
DRIVE_ROOT = "/content/drive/MyDrive/Quant_Research/strategy-research-agent/phase5c1_recovery"


In [ ]:
import os, sys, re, json, pathlib, subprocess
if MODE not in ("PREPARE", "PREFLIGHT", "EXECUTE"):
    raise RuntimeError("Unsupported mode")
if not re.fullmatch(r"[0-9a-f]{40}", EXECUTION_CODE_SHA):
    raise RuntimeError("A published reviewed execution SHA is required")
if NEW_AUTHORIZATION_ID != "RA-IBKR_TSM-SMA520_P5C1-0002":
    raise RuntimeError("Only the new registered recovery identity is supported")
if MODE == "EXECUTE" and (CONFIRM_AUTHORIZATION != NEW_AUTHORIZATION_ID or CONFIRM_REMOTE_CLAIM is not True):
    raise RuntimeError("Separate explicit new authorization and CLAIM confirmations required")
active = bool(GOVERNANCE_SHA or AUTHORIZATION_RECORD_SHA256) or MODE != "PREPARE"
if active and (not re.fullmatch(r"[0-9a-f]{40}", GOVERNANCE_SHA) or
               not re.fullmatch(r"[0-9a-f]{64}", AUTHORIZATION_RECORD_SHA256)):
    raise RuntimeError("New approved governance/record pins required; draft packet is not authority")
from google.colab import drive, userdata
drive.mount("/content/drive")
assert pathlib.Path("/content/drive/MyDrive").is_dir()
if active:
    os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")  # Never print or persist.
BASE = pathlib.Path("/content/c1-recovery-runtime")
BASE.mkdir(exist_ok=False)  # Never re-launch in this Runtime after an interruption.
subprocess.run([sys.executable,"-m","pip","install","uv==0.13.4"],check=True)
subprocess.run([sys.executable,"-m","uv","python","install","3.11.17"],check=True)
subprocess.run([sys.executable,"-m","uv","venv","--python","3.11.17",str(BASE/"venv")],check=True)
PYTHON = str(BASE/"venv/bin/python")
def checkout(repository, sha, name):
    target = BASE/name
    subprocess.run(["git","clone","https://github.com/hh4832/"+repository+".git",str(target)],check=True)
    subprocess.run(["git","-C",str(target),"checkout","--detach",sha],check=True)
    assert subprocess.check_output(["git","-C",str(target),"rev-parse","HEAD"],text=True).strip()==sha
    assert not subprocess.check_output(["git","-C",str(target),"status","--porcelain"],text=True).strip()
    return target
CODE = checkout("strategy-research-agent",EXECUTION_CODE_SHA,"code")
CORE = checkout("market-research-core",CORE_SHA,"core")
subprocess.run([sys.executable,"-m","uv","pip","sync","--python",PYTHON,
                str(CODE/"requirements/phase5c1-runtime.txt")],check=True)
subprocess.run([sys.executable,"-m","uv","pip","install","--python",PYTHON,"--no-deps","--no-build-isolation",
                "-e",str(CORE),"-e",str(CODE)],check=True)
assert subprocess.check_output([PYTHON,"-c","import sys; print(sys.version.split()[0])"],text=True).strip()=="3.11.17"
settings = dict(mode=MODE,code_sha=EXECUTION_CODE_SHA,drive_root=DRIVE_ROOT,
    recovery_archive=RECOVERY_ARCHIVE,original_preparation_archive=ORIGINAL_PREPARATION_ARCHIVE,
    preparation=PREPARATION,bind_url=BIND_URL,task_url=TASK_URL,
    confirm_authorization=CONFIRM_AUTHORIZATION,confirm_remote_claim=CONFIRM_REMOTE_CLAIM)
if active:
    GOV = checkout("strategy-research-agent",GOVERNANCE_SHA,"governance")
    settings.update(governance_root=str(GOV),execution_context=dict(authorization_id=NEW_AUTHORIZATION_ID,
        governance_sha=GOVERNANCE_SHA,record_sha256=AUTHORIZATION_RECORD_SHA256))
SETTINGS = BASE/"settings.json"
SETTINGS.write_text(json.dumps(settings))  # Pins/paths only; no credential.
sys.path.insert(0,str(CODE/"src"))
from strategy_agent.execution.c1_observe import run_logged
run_logged([PYTHON,"-u","-m","strategy_agent.execution.c1_recovery","--settings",str(SETTINGS)],
           cwd=CODE,parent=pathlib.Path(DRIVE_ROOT)/"runtime_logs"/NEW_AUTHORIZATION_ID,
           secrets=(os.environ.get("GITHUB_TOKEN",""),))


## Sequence and boundaries
Use a fresh CPU Runtime. Default PREPARE writes an unsigned new-authorization packet only: no data read,
no formal Task/Request, no approval, no CLAIM, no batch. The old authorization is permanently non-reusable.

1. Run default PREPARE once; review the new Drive authorization_preparations packet and prior-attempt disclosure.
2. A research lead must review the original Batch disposition and approve a NEW authorization through the existing
   append-only governance process. Record its new governance commit and record SHA. Do not copy old approvals.
3. In a fresh Runtime, set these two approved pins and the reviewed archive paths. PREPARE now enters the existing
   non-consuming authoritative pipeline and prints a NEW Request SHA, Task ID, preparation path and approval lines.
4. The lead personally posts distinct new BIND_AUTHORIZATION and APPROVE_TASK comments. In another fresh Runtime,
   set PREFLIGHT and those exact new subjects. Require the full live preflight PASS; this is not EXECUTE approval.
5. Only a separate explicit human decision may select EXECUTE with the new ID and both confirmations. No execution
   is authorized by this notebook's publication, this PR, a draft packet or CI success.

Read-only acceptance: inspect runtime_logs process_outcome.json and runner.log; verify PACKET_INDEX.json for draft
packets or use c1_archive.verify_archive on a completed formal archive. Confirm new auth/Task/Request/CLAIM identities,
Use/Budget/Task Events, receipt/output manifest, hash completeness and required outputs. A zero process exit alone is
not formal settlement or acceptance of performance. Never count prior failed partial output as accepted results.

Disconnection: NEVER press Run All/EXECUTE again. Read existing runtime_logs, durable attempts, Archive and server
CLAIM from another read-only session. If status is unknown, stop for reconciliation. There is no retry or resume of
an interrupted formal simulation. Drive is append-only by application rules, not physical WORM storage. CPU Runtime
wall time, RAM and Drive availability remain platform-dependent; review the documented feasibility risk before approval.
